# 文法誤り訂正を編集単位の F0.5 で評価する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/25_gec_education/gec_f05.ipynb)

NLP クックブック 25-05 ／ 文法誤り訂正と言語教育

GEC システムの出力を、「正しい編集をどれだけ提案できたか」で評価したい。再現率より適合率を重視する F0.5 を使う。

In [1]:
!pip install -q errant
!python -m spacy download en_core_web_sm -q

In [2]:
import errant

annotator = errant.load("en")

def edit_set(src, tgt):
    o, c = annotator.parse(src), annotator.parse(tgt)
    return {(e.o_start, e.o_end, e.c_str) for e in annotator.annotate(o, c)}

def gec_score(sources, hypotheses, references, beta=0.5):
    tp = fp = fn = 0
    for src, hyp, ref in zip(sources, hypotheses, references):
        h, r = edit_set(src, hyp), edit_set(src, ref)
        tp += len(h & r); fp += len(h - r); fn += len(r - h)
    p = tp / (tp + fp) if tp + fp else 1.0
    r = tp / (tp + fn) if tp + fn else 1.0
    f = (1 + beta ** 2) * p * r / (beta ** 2 * p + r) if p + r else 0.0
    return {"TP": tp, "FP": fp, "FN": fn, "P": round(p, 3), "R": round(r, 3), f"F{beta}": round(f, 3)}

src = ["I has went to school yesterday .", "She like apples very much .", "He go to the park every days ."]
ref = ["I went to school yesterday .", "She likes apples very much .", "He goes to the park every day ."]
systems = {
    "何もしない": src,
    "慎重 (1 か所だけ直す)": ["I went to school yesterday .", "She like apples very much .", "He go to the park every days ."],
    "積極的 (直しすぎ)": ["I went to the school yesterday .", "She likes apples a lot .", "He goes to a park every day ."],
    "完全": ref,
}
for name, hyp in systems.items():
    print(f"{name:20}", gec_score(src, hyp, ref))

何もしない                {'TP': 0, 'FP': 0, 'FN': 4, 'P': 1.0, 'R': 0.0, 'F0.5': 0.0}
慎重 (1 か所だけ直す)        {'TP': 1, 'FP': 0, 'FN': 3, 'P': 1.0, 'R': 0.25, 'F0.5': 0.625}
積極的 (直しすぎ)           {'TP': 4, 'FP': 4, 'FN': 0, 'P': 0.5, 'R': 1.0, 'F0.5': 0.556}
完全                   {'TP': 4, 'FP': 0, 'FN': 0, 'P': 1.0, 'R': 1.0, 'F0.5': 1.0}


- F0.5 は適合率を再現率の 2 倍重視する。間違った訂正を学習者に見せる害のほうが、見逃しより大きいという考え方。
- 正しい訂正は 1 通りではない。参照が 1 つだけだと、妥当な別解 (「very much → a lot」) が誤りとして数えられる。複数参照の場合は、文ごとに最も F 値が高くなる参照を選ぶ。
- 流暢性まで含めた訂正の評価には GLEU や、参照なしの指標 (IMPARA、SOME など) が使われる。